In [1]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q "reasoning-from-scratch>=0.1.2" "torch>=2.7.1" "tokenizers>=0.21.2" "sympy>=1.14.0" requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.6/64.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 107.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 128.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 150.3 MB/s eta 0:00:00


In [2]:
import torch
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import load_model_and_tokenizer

device = get_device()

model, tokenizer = load_model_and_tokenizer(which_model="base",device=device,use_compile=False)

Using NVIDIA CUDA GPU
qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [3]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)

prompt = render_prompt(raw_prompt)
torch.manual_seed(0)
response = generate_text_stream_concat_flex(
    model,tokenizer,prompt,device,
    max_new_tokens=2048,verbose=True,generate_func=generate_text_top_p_stream_cache,temperature=0.9,top_p=0.9
)

print(response)

 47 47


In [4]:
### laoding math dataset 

import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)

    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "math_full_minus_math500/refs/heads/main/"
        "math_full_minus_math500.json"
    )
    backup_url = (
        "https://f001.backblazeb2.com/file/reasoning-from-scratch/"
        "MATH/math_full_minus_math500.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
        except requests.RequestException:
            print("Using backup URL.")
            r = requests.get(backup_url, timeout=30)
            r.raise_for_status()

        data = r.json()

        if save_copy:
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data

math_train = load_math_train()
print("Dataset size:", len(math_train))

Dataset size: 12000


In [5]:
### rollout generation function

from reasoning_from_scratch.qwen3 import KVCache
from reasoning_from_scratch.ch04 import top_p_filter

@torch.no_grad()
def sample_response(
    model,tokenizer,prompt,device,max_new_tokens=2048,temperature=0.8,top_p=0.9
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt), device=device
        ).unsqueeze(0)
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids, cache=cache)[:, -1] # grab latest row of logits
    generated = []
    for _ in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature
        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(probas.cpu(), num_samples=1).to(device)
        token_id = next_token.item()
        generated.append(token_id)

        if tokenizer.eos_token_id is not None and token_id == tokenizer.eos_token_id:
            break
        logits = model(next_token, cache=cache)[:, -1]

    full_token_ids  = torch.cat([input_ids, torch.tensor(
        generated,device=device,dtype=input_ids.dtype
    ).unsqueeze(0)], dim=1)
    return full_token_ids,input_ids.numel(), tokenizer.decode(generated) # numel reutnrs product fo all dimensions of the tensor


In [6]:
torch.manual_seed(0)

token_ids,prompt_len,answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

 47<|endoftext|>


In [7]:
### rollout example 
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

In [8]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def reward_rlvr(answer_text, ground_truth):
    extracted = extract_final_candidate(
        answer_text, fallback=None
    )
    if not extracted:
        return 0.0
    correct = grade_answer(extracted, ground_truth)
    return float(correct)

In [ ]:
rollout_rewards = [reward_rlvr(answer_text, "83") for answer_text in rollouts]

In [10]:
from torch._tensor import Tensor

rewards: Tensor = torch.tensor(rollout_rewards,device=device)
advantages = (rewards - rewards.mean())/(rewards.std() + 1e-8)
print(rewards)
print(advantages)


tensor([1., 1., 0., 0.], device='cuda:0')
tensor([ 0.8660,  0.8660, -0.8660, -0.8660], device='cuda:0')


### Average log-probability of an answer

Given a prompt $x$ and an answer $y = (y_1, \dots, y_T)$, the model assigns a conditional probability to each answer token given everything before it. The average log-probability of the answer is:

$$
\bar{\ell}(y \mid x) = \frac{1}{T} \sum_{t=1}^{T} \log \pi_\theta\big(y_t \mid x, y_{<t}\big)
$$

where $\pi_\theta(\cdot \mid x, y_{<t})$ is the model's softmax distribution over the vocabulary at the position right before predicting token $y_t$. In code:

- `full_ids` = prompt tokens concatenated with answer tokens
- `logprobs[t_idx, next_tokens]` gathers $\log \pi_\theta(y_t \mid x, y_{<t})$ for every answer position $t$
- `torch.mean(...)` computes the average over $T$ answer tokens

This scalar is a length-normalized measure of how confident the model is in the answer it produced (or a candidate answer), and it is what later gets used (via its gradient) as the $\log \pi_\theta$ term in the GRPO policy-gradient loss.

In [ ]:
@torch.inference_mode()
def avg_logprob_answer(model,tokenizer,prompt,answer,device="cpu"):
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids+answer_ids,device=device)
    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)
    

    start = len(prompt_ids) -1
    end = full_ids.shape[0] -1
    t_idx = torch.arange(start, end, device=device)
    next_tokens = full_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]
    return torch.mean(next_token_logps).item()

In [12]:
avg_logprob_val = avg_logprob_answer(
model, tokenizer,
prompt=prompt,
answer=answer_text,
device=device)
print(avg_logprob_val)

-2.171875


In [13]:
sequence_logprob_val = avg_logprob_val * (
len(tokenizer.encode(answer_text))
)
print(sequence_logprob_val)

-8.6875


In [14]:
def sequence_logprob_draft(model, token_ids, prompt_len):
    # token_ids from sample_response is already batched, shape (1, seq_len)
    logits = model(token_ids).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    token_ids = token_ids.squeeze(0)

    # Positions whose next-token probabilities we want to predict
    start = prompt_len - 1
    end = token_ids.shape[0] - 1
    t_idx = torch.arange(start, end, device=token_ids.device)
    next_tokens = token_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Sum log probabilities over the answer tokens.
    return torch.sum(next_token_logps)

In [15]:
print(sequence_logprob_draft(model, token_ids, prompt_len))

tensor(-8.7035, device='cuda:0', grad_fn=<SumBackward0>)


### GPU-friendlier version with `torch.gather`

`sequence_logprob_draft` builds `t_idx` with `torch.arange` and then indexes `logprobs[t_idx, next_tokens]` — two separate advanced-indexing tensors that PyTorch has to broadcast against each other. We can replace that with a single `torch.gather` call, which maps directly onto one fused CUDA gather kernel instead of constructing and broadcasting an extra index tensor:

- `logprobs[:-1]` — drop the last row, since the token at the last position has no "next token" to score
- `token_ids[1:].unsqueeze(-1)` — the target token at each position, reshaped to `(seq_len-1, 1)` so it lines up with `gather`'s `index` argument
- `.gather(1, ...)` — for each row, pick out the log-probability of that row's target token along the vocab dimension
- `selected[prompt_len - 1:]` — keep only the answer positions (the prompt tokens have no reward-relevant log-prob to sum)

In [16]:
def sequence_logprob(model, token_ids, prompt_len):
    # token_ids from sample_response is already batched, shape (1, seq_len)
    logits = model(token_ids).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    token_ids = token_ids.squeeze(0)

    selected = logprobs[:-1].gather(
        1, token_ids[1:].unsqueeze(-1)
    ).squeeze(-1)
    return torch.sum(selected[prompt_len - 1:])

In [17]:
print(sequence_logprob(model, token_ids, prompt_len))

tensor(-8.7035, device='cuda:0', grad_fn=<SumBackward0>)


In [ ]:
def compute_grpo_loss(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
):
    assert num_rollouts >= 2
    roll_logps, roll_rewards, samples = [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    for _ in range(num_rollouts):
        # Stage 1: generate rollouts
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        # Stage 2: compute rewards
        reward = reward_rlvr(text, example["answer"])

        # Stage 4: compute logprobs
        logp = sequence_logprob(model, token_ids, prompt_len)

        roll_logps.append(logp)
        roll_rewards.append(reward)
        samples.append(
            {
                "text": text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training:
        model.train()

    # Stage 2: collect all rewards
    rewards = torch.tensor(roll_rewards, device=device)

    # Stage 3: compute advantages
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # Stage 4: collect all logprobs
    logps = torch.stack(roll_logps)

    # Stage 5: compute policy gradient loss
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss  # In the next chapter we add a KL term here

    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

In [19]:
torch.manual_seed(123)
stats = compute_grpo_loss(
model=model,
tokenizer=tokenizer,
example=math_train[4],
device=device,
num_rollouts=2,
max_new_tokens=256,
temperature=0.8,
top_p=0.9
)
from pprint import pprint
pprint(stats)

{'advantages': [0.0, 0.0],
 'loss': -0.0,
 'loss_tensor': tensor(-0., device='cuda:0', grad_fn=<NegBackward0>),
 'pg_loss': -0.0,
 'rewards': [0.0, 0.0],
 'samples': [{'gen_len': 4, 'reward': 0.0, 'text': ' 14<|endoftext|>'},
             {'gen_len': 256,
              'reward': 0.0,
              'text': ' 4\n'
                      "To solve the problem, let's break it down step by "
                      'step:\n'
                      '\n'
                      '1. **Define Variables:**\n'
                      '   - Let \\( x \\) be the number of days Sam works.\n'
                      '   - Then, the number of days he does not work is \\( '
                      '20 - x \\).\n'
                      '\n'
                      '2. **Set Up the Earnings Equation:**\n'
                      '   - For each day he works, he earns \\$60.\n'
                      '   - For each day he does not work, he loses \\$30.\n'
                      '   - His total earnings are \\$660.\n'
      

In [20]:
import time

def train_rlvr_grpo(
    model,
    tokenizer,
    math_data,
    device,
    steps=None,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=50,
    checkpoint_dir=".",
    csv_log_path=None,
):
    if steps is None:
        steps = len(math_data)

    # Stage 1: Initialize optimizer
    # (model was already initialized outside the function)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    current_step = 0

    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        csv_log_path = f"train_rlvr_grpo_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    try:
        # Stage 2: Iterate over training steps
        for step in range(steps):

            # Stage 3: Reset loss gradient (best practice
            # to do this at the beginning of each step)
            optimizer.zero_grad()

            current_step = step + 1
            example = math_data[step % len(math_data)]

            # Stage 4: Calculate GRPO loss
            stats = compute_grpo_loss(
                model=model,
                tokenizer=tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )

            # Stage 5: Backward pass to calculate loss gradients
            stats["loss_tensor"].backward()

            # Clip large gradients to improve training stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Stage 6: Update model weights using loss gradients
            optimizer.step()

            # Stage 7: Collect rewards, average response lengths,
            # and losses
            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            step_tokens = sum(
                sample["gen_len"] for sample in stats["samples"]
            )
            avg_response_len = (
                step_tokens / len(stats["samples"]) if stats["samples"] else 0.0
            )
            append_csv_metrics(
                csv_log_path, current_step, steps,
                stats["loss"], reward_avg, avg_response_len,
            )

            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_resp_len={avg_response_len:.1f}"
            )

            # Stage 8: Save model checkpoint
            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model=model,
                    checkpoint_dir=checkpoint_dir,
                    step=current_step,
                )
                print(f"Saved checkpoint to {ckpt_path}")

    except KeyboardInterrupt:
        # Save a model checkpoint if we interrupt training early
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=max(1, current_step),
            suffix="interrupt",
        )
        print(f"\nKeyboardInterrupt. Saved checkpoint to {ckpt_path}")
        return model

    return model

In [21]:
def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = (
        checkpoint_dir /
        f"qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.pth"
    )
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path

In [22]:
# Utility function to save the results to a CSV file
def append_csv_metrics(
    csv_log_path, step_idx, total_steps,
    loss, reward_avg, avg_response_len,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "step,total_steps,loss,reward_avg,avg_response_len\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{step_idx},{total_steps},{loss:.6f},{reward_avg:.6f},"
            f"{avg_response_len:.6f}\n"
        )

In [23]:
device = get_device()
model.to(device)
torch.manual_seed(0)
train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=50,
    num_rollouts=4,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=5,
    checkpoint_dir=".",
)

Using NVIDIA CUDA GPU
[Step 1/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=385.0
[Step 2/50] loss=-11.6029 reward_avg=0.250 avg_resp_len=133.8
[Step 3/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=14.2
[Step 4/50] loss=0.4901 reward_avg=0.250 avg_resp_len=6.2
[Step 5/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=6.8
Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00005.pth
[Step 6/50] loss=-1.0259 reward_avg=0.500 avg_resp_len=9.2
[Step 7/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=17.2
[Step 8/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=9.5
[Step 9/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=7.0
[Step 10/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=10.0
Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00010.pth
[Step 11/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=6.2
[Step 12/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=6.0
[Step 13/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=6.5
[Step 14/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=6.0
[Step 15/50] loss=-0.0000 rewa

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)